# 第V部：Euler方程式の時間積分

**問い：同じ空間離散化を、前進Euler・SSP-RK2・SSP-RK3で時間発展させると、誤差・安定性・計算量はどう変わるか。**

Roe・LLF (Rusanov)・HLL・HLLCごとに時間積分だけを切り替えます。時間精度の測定と、衝撃波を含む実用問題の比較は別の実験として扱います。

[I 移流・Burgers](01_advection_burgers.ipynb) · [II Eulerの古典的解法](02_euler_classical.ipynb) · [III 数値流束](03_euler_riemann_solvers.ipynb) · [IV 空間再構築](04_spatial_reconstruction.ipynb) · [V 時間積分](05_time_integration.ipynb)

## 0. 設定と実行方法

`notebooks/` と `src/` の配置を保ち、設定セルを確認して上から実行します。
数値流束・初期条件・境界条件・厳密解は `euler1d_core.py`、空間と時間を独立に選ぶ計算は `euler1d_mol.py`、実験・描画は `euler1d_experiments.py` を使用します。
図のラベルは英語、解説は日本語です。物理量はすべて無次元です。
動画は標準で有効です。Shu–Osherの高解像度参照計算は設定で有効にします。
設定を変更したらカーネルを再起動し、全セルを実行してください。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/euler1d_mol.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("リポジトリ一式の notebooks/ と src/ の配置を保ってください。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import euler1d_core as core
import euler1d_mol as mol
import euler1d_experiments as ex
from euler1d_study import show_table, shu_reference

ex.CACHE.clear()
TABLES = {}
reference = None
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("MOL checks:", len(mol.run_tests()))

### 時間積分・空間離散化の設定

| 設定値 | 内容 | 1ステップの右辺評価数 |
|:--|:--|:--|
| `forward_euler` | 前進Euler、時間1次 | 1 |
| `ssprk2` | 2段階SSP-RK2、時間2次 | 2 |
| `ssprk3` | 3段階SSP-RK3、時間3次 | 3 |

`fluxes` は `Roe`、`LLF`、`HLL`、`HLLC` から選びます。
`reconstruction` は `constant` または `muscl`、`limiter` は `minmod`、`mc`、`unlimited` です。`constant` ではリミッターは使いません。

**2つの実験を区別します。**

| 実験 | 空間設定 | 時間刻み |
|:--|:--|:--|
| 時間収束試験 | `temporal_reconstruction`・`temporal_limiter`、固定格子`temporal_n` | 全方式で同じ $\Delta t=T/N_t$。`temporal_steps`で$N_t$を指定 |
| 接触面・衝撃波・動画 | `reconstruction`・`limiter`、`n_basic` | 共通のCFL規則。各方式で状態が異なるので実際の刻みは同一とは限らない |

時間収束試験の標準は区分一定です。非滑らかな勾配制限の切替を避けて時間次数を確認し、その後にMUSCLでも調べます。
`reference_steps` は同じ半離散系をSSP-RK3で積分する2つの細かいステップ数です。基準は同じ格子・同じ流束・同じ再構築・同じ初期値で計算します。

### 動画の選択肢

| `movie_problem` | 内容 |
|:--|:--|
| `entropy` | 滑らかなエントロピー波の移流 |
| `contact` | 移動する接触不連続面 |
| `contact_static` | 静止する接触不連続面 |
| `sod` | Sod衝撃波管 |
| `double_rarefaction` | 二重希薄波。負圧などで停止した方式は停止後に描かない |
| `lax` | Lax衝撃波管 |
| `transonic` | 音速をまたぐ希薄波 |

`movie_fluxes` は `("Roe", "LLF", "HLL", "HLLC")` の一部または全部を指定します。各流束に1本の動画を作ります。標準は `("HLLC",)` です。
`movie_field` は `"rho"`（密度）、`"u"`（速度）、`"p"`（圧力）、`"e"`（比内部エネルギー）です。
`movie_frames` は表示フレーム数であり、数値積分のステップ数ではありません。
`shu_osher` はこの動画機能の対象外です。動画を省略する場合は `run_movies=False` にします。

In [ ]:
CFG = {
    "gamma": 1.4,
    "fluxes": ("Roe", "LLF", "HLL", "HLLC"),
    "time_integrators": ("forward_euler", "ssprk2", "ssprk3"),
    "reconstruction": "muscl",  # 接触面・衝撃波・動画で固定
    "limiter": "minmod",
    "n_basic": 200,
    "cfl": 0.4,
    "wave_speeds": "einfeldt",
    "roe_entropy": 0.1,
    "temporal_reconstruction": "constant",  # 時間収束試験の空間設定
    "temporal_limiter": "minmod",  # constantでは使用しない
    "temporal_n": 48,
    "temporal_t_end": 0.2,
    "temporal_steps": (40, 80, 160, 320),
    "reference_steps": (1280, 2560),
    "fixed_cfl_limit": 0.9,  # 超過時は停止。固定dtを自動変更しない
    "run_time_convergence": True,
    "run_robustness": True,
    "run_extras": False,
    "run_shu_osher": False,
    "n_shu_osher": 400,
    "n_reference": (1600, 3200),
    "reference_limiter": "mc",
    "run_movies": True,
    "movie_problem": "sod",  # 選択肢は直前の表を参照
    "movie_fluxes": ("HLLC",),
    "movie_field": "rho",
    "movie_frames": 41,
    "export_results": True,
}
ex.validate_config(CFG, "time")
OUTPUT_DIR = ROOT / "five_part_outputs" / "temporal"
print(CFG)

## 1. 数値流束・空間再構築・時間積分を分ける

$U=(\rho,\rho u,E)^T$、$F(U)=(\rho u,\rho u^2+p,u(E+p))^T$、$p=(\gamma-1)(E-\rho u^2/2)$とします。
未知量はセル平均 $\bar U_i$ です。境界の左右状態を再構築し、数値流束 $\widehat F$ を求めると、

$$\frac{d\bar U_i}{dt}=\mathcal L(\bar U)_i=-\frac{\widehat F_{i+1/2}-\widehat F_{i-1/2}}{\Delta x}.$$

これは**空間だけを離散化した常微分方程式系（半離散系）**です。数値流束と再構築を固定すると右辺 $\mathcal L$ が決まり、その後に時間積分法を選びます。

この部で使うRoe・LLF (Rusanov)・HLL・HLLCの流束は、$\Delta t$に依存しません。古典的LFとRichtmyer-LWは時間刻みを流束の構成に用いるため、ここで自由に組み合わせる対象には含めません。
HLLは標準でEinfeldt波速を用い、図ではHLL (HLLE)と表示します。Roeのエントロピー補正、外側波速、初期条件、境界条件は比較中に揃えます。[R1–R3]

### 誤差・保存則・失敗判定

保存変数をセル平均した厳密解を $\bar U_{\rm exact}$ とし、密度の平均絶対誤差を

$$E_{\rm total}=\frac1N\sum_i|\bar\rho_i-\bar\rho_{{\rm exact},i}|$$

で測ります。図の黒線は点値の厳密解、誤差表はセル平均基準です。Shu–Osherだけは高解像度の数値参照解を使います。
周期境界では保存量の総和を、開いた境界では数値流束の積算を差し引いた残差を測定します。残差が小さいことと誤差が小さいことは別です。

負の密度・圧力、非有限値、指定された固定時間刻みのCFL上限違反は停止として記録します。下限値への置換、刻み直し、別流束への自動切替は行いません。停止した途中時刻の解を最終時刻の解と重ねません。
CFL数を一定にしても、状態依存の波速のため実際の時間刻みが全方式で同一とは限りません。

## 2. 3つの更新式

以下は時間非依存の半離散系 $dU/dt=\mathcal L(U)$ です。$\mathcal E(U)=U+\Delta t\mathcal L(U)$ とおきます。

**前進Euler**
$$U^{n+1}=\mathcal E(U^n).$$

**SSP-RK2（2段階・2次）**
$$U^{(1)}=\mathcal E(U^n),\qquad U^{n+1}=\tfrac12U^n+\tfrac12\mathcal E(U^{(1)}).$$

**SSP-RK3（3段階・3次、Shu–Osher型）**
$$U^{(1)}=\mathcal E(U^n),\qquad U^{(2)}=\tfrac34U^n+\tfrac14\mathcal E(U^{(1)}),$$
$$U^{n+1}=\tfrac13U^n+\tfrac23\mathcal E(U^{(2)}).$$

単に「RK2」「RK3」と呼ぶのではなく、使う公式を特定します。各段階で境界条件・再構築・流束を求め直します。[R8, T1, T2]

SSPは、前進Eulerが所定の時間刻み制限下で持つ強安定性を、凸結合によって保つ考え方です。**前進Euler＋選んだ空間離散化にその性質がなければ、SSPという名前だけでは保証されません。** 成分別MUSCLを含むEuler系の任意の状態について、TVD性・正値性を保証する主張ではありません。

### 境界流束も同じ重みで積分する

段階の正味流入を $B^{(k)}=F^{(k)}_{\rm left}-F^{(k)}_{\rm right}$ とすると、境界寄与は前進Eulerで $\Delta t B^{(1)}$、SSP-RK2で $\Delta t(B^{(1)}+B^{(2)})/2$、SSP-RK3で

$$\Delta t\left(\tfrac16B^{(1)}+\tfrac16B^{(2)}+\tfrac23B^{(3)}\right)$$

です。この重みと保存変数の更新を揃えて、開いた境界での保存収支を検査します。
この教材の境界条件と流束は時間に明示的に依存しません。時間依存の外力や境界を加える場合は段階時刻も指定する必要があります。

## 3. 時間誤差と空間誤差を分ける

連続方程式の厳密解と比較した誤差 $E_{\rm total}$ には空間離散化誤差も含まれます。時間刻みを小さくしても、この誤差が一定値へ近づくことがあります。

時間積分だけを見るため、**同じ半離散系**を非常に細かい時間刻みで積分した数値参照解 $U_{\rm ref}$ と比較します。

$$E_t(\Delta t)=\frac1N\sum_i|\rho_{\Delta t,i}(T)-\rho_{{\rm ref},i}(T)|,$$
$$p_t=\frac{\log(E_t(\Delta t_1)/E_t(\Delta t_2))}{\log(\Delta t_1/\Delta t_2)}.$$

参照解は各流束について別々に計算します。細かい**空間格子**の解を使うのではありません。
2つの細かい時間刻みの参照解の差 `reference_gap` と、測定した最小誤差との比 `gap_over_error` も表示します。比が0.05以上なら参照解不足の警告を出します。この判定だけで参照解の誤差を厳密に保証したことにはなりません。

固定ステップ数 $N_t$ から $\Delta t=T/N_t$ を決め、全時間積分法で同じ刻みを使います。CFL上限を超える設定は停止し、勝手に刻みを変更しません。
`temporal_n`や終了時刻を大きくしたときは、`temporal_steps`と`reference_steps`も十分大きくしてください。

**予想：** 滑らかな半離散解に対して、時間積分次数は1・2・3です。ただし、勾配制限などで右辺の滑らかさが失われる場合や丸め誤差・参照誤差が支配する場合は、この次数がそのまま観測されるとは限りません。

In [ ]:
if CFG["run_time_convergence"]:
    time_rows, reference_rows = ex.temporal_convergence(CFG)
    TABLES["temporal_convergence"] = time_rows
    TABLES["temporal_reference_checks"] = reference_rows
    TABLES["cost_at_tolerance"] = ex.cost_table(time_rows)

### 結果の読み方

`E_time`は同じ空間離散化の参照解との差、`E_total`は連続方程式の厳密解との差です。両者の大小・時間刻み依存性を比較してください。
`n_rhs`は右辺評価回数で、途中で失敗した試行も数えます。`seconds`はこの環境での実測時間であり、短い計算ではばらつきます。
コスト表は**計算済みの候補の中で**誤差目標を満たす最小`n_rhs`を選びます。`None`は未達です。すべての刻みを探索した最適コストではありません。

**確認：** SSP-RK3の`E_time`が小さくても、`E_total`は必ずSSP-RK2より小さくなるでしょうか。誤差の相殺と空間誤差の下限を区別してください。

## 4. 安定性と精度は別の性質

試験方程式 $y'=\lambda y$ に対し、$y^{n+1}=R(z)y^n$、$z=\lambda\Delta t$と書けます。

$$R_{\rm FE}=1+z,\qquad R_{\rm RK2}=1+z+\tfrac12z^2,\qquad R_{\rm RK3}=1+z+\tfrac12z^2+\tfrac16z^3.$$

純虚数 $z=i\beta$ に限れば、前進EulerとSSP-RK2は$\beta\ne0$で$|R|>1$、SSP-RK3は$|\beta|\le\sqrt3$に安定な区間を持ちます。
これは**純虚数軸上の線形安定性**の比較です。散逸を含む実際の空間離散化の固有値は一般に純虚数ではなく、下図だけから非線形Euler系の安定CFL数を決めることはできません。[T1]

In [ ]:
beta = np.linspace(0.0, 2.2, 301)
z = 1j * beta
amplification = (np.abs(1 + z), np.abs(1 + z + z**2 / 2), np.abs(1 + z + z**2 / 2 + z**3 / 6))
fig = plt.figure(figsize=(8.0, 4.4))
ax = fig.gca()
for j, ti in enumerate(mol.TIME_INTEGRATORS):
    ax.plot(beta, amplification[j], color=ex.COLORS[j], ls=ex.STYLES[j], label=mol.TIME_LABELS[ti])
ax.axhline(1.0, color="black", lw=0.8)
ax.set(xlabel=r"$\beta=\mathrm{Im}(\lambda)\Delta t$", ylabel=r"$|R(i\beta)|$", title="Linear stability on the imaginary axis")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)

## 5. 同じMUSCLでの滑らかな波の移流

ここからは設定のreconstructionとlimiterを固定し、時間積分を変えます。標準はMUSCL＋minmodです。CFL規則は共通ですが、実際の時間刻みが同一とは限りません。

In [ ]:
for flux in CFG["fluxes"]:
    runs = ex.group_runs("entropy", flux, CFG, "time")
    ex.plot_group(runs, "time", fields=("rho",))
    ex.summarize(runs)

## 6. 接触不連続面

低散逸の空間再構築に前進Eulerを組み合わせたとき、振動や拡散がどのように現れるかを確認します。滑らかな問題で測った次数を、不連続面での局所次数と混同しないでください。

In [ ]:
for flux in CFG["fluxes"]:
    runs = ex.group_runs("contact", flux, CFG, "time")
    ex.plot_group(runs, "time", fields=("rho",))
    ex.summarize(runs)

## 7. Sod衝撃波管

同じ流束・再構築で時間積分だけを変えます。途中段階の負圧などで停止した場合も結果として記録します。

In [ ]:
for flux in CFG["fluxes"]:
    runs = ex.group_runs("sod", flux, CFG, "time")
    ex.plot_group(runs, "time", fields=("rho", "p"))
    ex.summarize(runs)

## 8. 二重希薄波と停止判定

`run_robustness=True`で低密度・低圧力の問題を比較します。破綻しないことは精度の高さと同じではありません。
停止した方式の途中状態は最終時刻の図へ描かず、表の`t_reached`と停止理由を確認します。

In [ ]:
if CFG["run_robustness"]:
    for flux in CFG["fluxes"]:
        runs = ex.group_runs("double_rarefaction", flux, CFG, "time")
        ex.plot_group(runs, "time", fields=("rho", "p"))
        ex.summarize(runs)

## 9. 時間積分法を比較する動画

標準はHLLC・MUSCL＋minmodのSod問題です。流束や問題を設定セルで変更できます。
時間積分の刻みと動画のフレーム時刻は区別します。補間は表示専用で、計算停止後の解は外挿しません。

In [ ]:
if CFG["run_movies"]:
    for flux in CFG["movie_fluxes"]:
        fig, movie = ex.comparison_movie(CFG["movie_problem"], flux, CFG, "time")
        display(HTML(movie.to_jshtml()))
        plt.close(fig)

## 10. 総合演習：数値流束・空間・時間の組合せ

まずHLLCを固定して、区分一定＋前進Euler、MUSCL＋SSP-RK2、MUSCL＋SSP-RK3を比較します。
**最初の2構成では空間と時間を同時に変えています。** これは一要素だけの比較ではなく、第III～V部で個別に学んだ要素を組み合わせた総合評価です。
次に、MUSCL＋minmod＋SSP-RK2を固定して4つの数値流束を比べます。

SSP-RK3を使っても、空間2次のMUSCLと組み合わせた全体が3次精度になるわけではありません。

In [ ]:
combined = [
    ex.get_run("contact", "HLLC", CFG, reconstruction="constant", time_integrator="forward_euler"),
    ex.get_run("contact", "HLLC", CFG, reconstruction="muscl", limiter="minmod", time_integrator="ssprk2"),
    ex.get_run("contact", "HLLC", CFG, reconstruction="muscl", limiter="minmod", time_integrator="ssprk3"),
]
# 各線の空間・時間の組合せを明示するため、完全なラベルを用いる。
fig = plt.figure(figsize=(11.5, 4.8))
ax = fig.add_axes((0.08, 0.16, 0.51, 0.70))
xp = np.linspace(0.0, 1.0, 1601)
ax.plot(xp, core.exact_pointwise(core.PROBLEMS["contact"], xp, 0.25, CFG["gamma"])[:, 0], color="black", label="Exact")
for j, run in enumerate(combined):
    if run.status == "ok":
        ax.plot(run.x, run.U[:, 0], color=ex.COLORS[j], ls=ex.STYLES[j], label=mol.reconstruction_label(run.reconstruction, run.limiter) + " + " + mol.TIME_LABELS[run.time_integrator])
ax.set(xlabel="$x$", ylabel="Density", xlim=(0.4, 0.8), title="HLLC | combined configurations")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=9)
plt.show()
plt.close(fig)
ex.summarize(combined)

# 最後は空間・時間を固定し、数値流束の違いを比較する。
from euler1d_study import plot_profiles
flux_comparison = [ex.get_run("sod", flux, CFG, reconstruction="muscl", limiter="minmod", time_integrator="ssprk2") for flux in CFG["fluxes"]]
plot_profiles(flux_comparison, title_suffix="Fixed MUSCL-minmod + SSP-RK2")
ex.summarize(flux_comparison)

## 11. 発展実験と結果保存

追加実験はLax衝撃波管、Shu–Osherは2解像度の数値参照解を伴う比較です。
時間収束表・参照解の差・コスト表は、数値解とともに保存します。

**発展課題：** 空間再構築をMCや制限なしに替えて、同じ時間刻みが適切か確認してください。計算が発散した場合、時間積分の次数不足と安定性の問題を分けて説明してください。

In [ ]:
if CFG["run_extras"]:
    for flux in CFG["fluxes"]:
        runs = ex.group_runs("lax", flux, CFG, "time")
        ex.plot_group(runs, "time")
        ex.summarize(runs)
if CFG["run_shu_osher"]:
    reference = shu_reference(CFG)
    for flux in CFG["fluxes"]:
        runs = ex.group_runs("shu_osher", flux, CFG, "time", n=CFG["n_shu_osher"])
        ex.plot_group(runs, "time", reference=reference)
        ex.summarize(runs, reference=reference)
if CFG["export_results"]:
    saved_rows = ex.export_results(OUTPUT_DIR, CFG, TABLES, reference=reference)

## まとめ

数値流束は境界を通る流れを、空間再構築はその左右状態を、時間積分は半離散系の時間発展を決めます。
方法名だけでなく、波速推定・エントロピー補正・再構築変数・リミッター・時間積分公式・CFL規則を揃えて比較してください。

時間次数の測定には同じ空間離散化の参照解を使い、実用問題の評価には厳密解との全誤差・頑健性・計算量を併記します。衝撃波を含む問題で一つの収束次数だけを指標にしないことも重要です。
本教材は1次元・一様格子・単一理想気体の学習用実装です。包括的な正値性保証、真空、重力、冷却、MHD、多次元は対象外です。

## 参考文献・実装の確認先

以下は原論文または開発者による公式資料です。Notebookのコードはこれらの数式・考え方を参考に、本教材用に組み立てた実装であり、Clawpack等を呼び出してはいません。

**[R1]** Clawpack, *The Euler equations of gas dynamics*：波の構造と厳密リーマン解。  
https://www.clawpack.org/riemann_book/html/Euler.html

**[R2]** Clawpack, *Approximate solvers for the Euler equations of gas dynamics*：Roe平均、HLLEの波速推定、低密度問題。  
https://www.clawpack.org/riemann_book/html/Euler_approximate.html

**[R3]** Clawpack, *Riemann Solver Package — Euler Equations*：HLLCの接触波速度と中間状態。  
https://www.clawpack.org/pyclaw/rp.html#euler-equations

**[R4]** FLASH User's Guide, *Hydrodynamics Test Problems*, §35.1.1：Sod問題と波面の位置。  
https://flash.rochester.edu/site/flashcode/user_support/flash_ug_devel/node191.html

**[R5]** Clawpack Gallery, *Shu–Osher problem*：初期状態、領域、終了時刻。  
https://www.clawpack.org/gallery/pyclaw/gallery/shocksine.html

**[R6]** Toro, E. F., Spruce, M., & Speares, W. (1994), *Restoration of the contact surface in the HLL-Riemann solver*, Shock Waves, 4, 25–34.  
https://doi.org/10.1007/BF01414629

**[R7]** Harten, A. (1983), *High resolution schemes for hyperbolic conservation laws*, Journal of Computational Physics, 49, 357–393.  
https://doi.org/10.1016/0021-9991(83)90136-5

**[R8]** Shu, C.-W., & Osher, S. (1988), *Efficient implementation of essentially non-oscillatory shock-capturing schemes*, Journal of Computational Physics, 77, 439–471.  
https://doi.org/10.1016/0021-9991(88)90177-5

**[R9]** Shu, C.-W., & Osher, S. (1989), *Efficient implementation of essentially non-oscillatory shock-capturing schemes, II*, Journal of Computational Physics, 83, 32–78.  
https://doi.org/10.1016/0021-9991(89)90222-2

初期条件・波速推定・エントロピー補正・リミッター・CFL数まで含めて、比較の設定を報告することが重要です。
「Roe」や「HLLC」という名前だけでは、実装のすべては決まりません。

**[T1]** Gkeyll developers, *Strong-Stability preserving Runge-Kutta time-steppers*：SSP-RK2・3段階SSP-RK3の更新式。
https://gkeyll.readthedocs.io/en/latest/dev/ssp-rk.html

**[T2]** Gottlieb, S., Shu, C.-W., & Tadmor, E. (2001), *Strong Stability-Preserving High-Order Time Discretization Methods*, SIAM Review, 43, 89–112.
https://doi.org/10.1137/S003614450036757X